In [1]:

from google.colab import files

uploaded = files.upload()


Saving archive (2).zip to archive (2).zip


In [2]:

import zipfile
import os

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/Dataset')

print("Images uploaded successfully!")


Images uploaded successfully!


In [3]:

import tensorflow as tf

train_data = tf.keras.utils.image_dataset_from_directory(
    '/content/Dataset',
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(128, 128),
    batch_size=32
)

test_data = tf.keras.utils.image_dataset_from_directory(
    '/content/Dataset',
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(128, 128),
    batch_size=32
)

print("Classes:", train_data.class_names)


Found 10088 files belonging to 4 classes.
Using 8071 files for training.
Found 10088 files belonging to 4 classes.
Using 2017 files for validation.
Classes: ['Cat', 'Dog', 'test_set', 'training_set']


In [6]:

from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128, 128, 3)),

    # First CNN layer
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Second CNN layer
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Third CNN layer
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=5
)


Epoch 1/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 253s 1000ms/step - accuracy: 0.0033 - loss: -1536982501031936.0000 - val_accuracy: 0.0015 - val_loss: -2376834364735488.0000
Epoch 2/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 257s 982ms/step - accuracy: 0.0033 - loss: -3680169839558656.0000 - val_accuracy: 0.0015 - val_loss: -5258992032940032.0000
Epoch 3/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 264s 990ms/step - accuracy: 0.0033 - loss: -7522816681312256.0000 - val_accuracy: 0.0015 - val_loss: -10164485708840960.0000
Epoch 4/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 262s 990ms/step - accuracy: 0.0033 - loss: -13743222111076352.0000 - val_accuracy: 0.0015 - val_loss: -17808328126103552.0000
Epoch 5/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 259s 977ms/step - accuracy: 0.0033 - loss: -23107050754015232.0000 - val_accuracy: 0.0015 - val_loss: -28988585411608576.0000


In [9]:

from google.colab import files
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np

uploaded = files.upload()

image_path = list(uploaded.keys())[0]

img = load_img(image_path, target_size=(128, 128))
img = img_to_array(img) / 255.0
img = np.expand_dims(img, axis=0)

prediction = model.predict(img)[0][0]

if prediction >= 0.5:
    print("Prediction: Dog 🐶")
else:
    print("Prediction: Cat 🐱")


Saving download (1).jfif to download (1) (1).jfif
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
Prediction: Dog 🐶
